# 05 - Demo: Open-Vocabulary 3D Object Search

This notebook runs the qualitative demo on a Polycam room:

1. Load a processed point cloud
2. Extract real Concerto features
3. Apply the trained MLP translation head
4. Query the scene with free text and visualize a heatmap


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
BRANCH = 'dev/eval-demo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

%cd {REPO_DIR}
!git restore configs/train_mlp_s3dis.yaml notebooks/pyproject.toml
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull --no-edit origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

!git clone https://github.com/Pointcept/Concerto.git /content/Concerto || true
sys.path.insert(0, REPO_DIR)


In [ ]:
%cd /content/Deep_learning_project/notebooks

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f'HF_TOKEN={HF_TOKEN}\n')

!uv python install 3.10.12
!uv sync --python 3.10.12
!pip install -q open_clip_torch plotly

## 1. Load the Polycam Scan

In [ ]:
import numpy as np
import plotly.graph_objects as go
from src.visualize import visualize_point_cloud

POLYCAM_DIR = Path('/content/drive/MyDrive/DL_Project/data/polycam_processed/room1')
CHECKPOINT_NAME = 'last_model.pth'
CHECKPOINT = Path(f'/content/drive/MyDrive/DL_Project/checkpoints/mlp_s3dis/{CHECKPOINT_NAME}')
CACHE_DIR = Path('/content/demo_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_CLIP_PATH = CACHE_DIR / f'room1_features_clip_{CHECKPOINT.stem}.npy'

coord = np.load(POLYCAM_DIR / 'coord.npy').astype(np.float32)
color = np.load(POLYCAM_DIR / 'color.npy').astype(np.float32)
normal = np.load(POLYCAM_DIR / 'normal.npy').astype(np.float32)

print(f'Loaded scan: {len(coord):,} points')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Feature cache: {FEATURES_CLIP_PATH}')

In [ ]:
fig = visualize_point_cloud(
    points=coord,
    colors=color,
    title='Polycam Scan - Raw Point Cloud',
    point_size=1.5,
)
fig.show()

## 2. Build Real CLIP-Space Point Features

This cell runs Concerto plus the trained translation head inside the `uv` environment, then saves per-point CLIP-space features to a local cache file.

In [ ]:
!CHECKPOINT_PATH={CHECKPOINT} FEATURES_CLIP_PATH={FEATURES_CLIP_PATH} POLYCAM_DIR={POLYCAM_DIR} PYTHONPATH=/content/Deep_learning_project:/content/Concerto CONCERTO_DIR=/content/Concerto uv run python - <<'PY'
import os
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path

from src.encoder import ConcertoEncoder
from src.translation_head import MLPTranslationHead

polycam_dir = Path(os.environ['POLYCAM_DIR'])
checkpoint_path = Path(os.environ['CHECKPOINT_PATH'])
out_path = Path(os.environ['FEATURES_CLIP_PATH'])

coord = np.load(polycam_dir / 'coord.npy').astype(np.float32)
color = np.load(polycam_dir / 'color.npy').astype(np.float32)
normal = np.load(polycam_dir / 'normal.npy').astype(np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

encoder = ConcertoEncoder(device=device)
encoder.eval()

with torch.no_grad():
    features_3d = encoder({'coord': coord, 'color': color, 'normal': normal})
print('3D features shape:', tuple(features_3d.shape))

checkpoint = torch.load(checkpoint_path, map_location=device)
model_cfg = checkpoint['config']['model']
mlp = MLPTranslationHead(
    input_dim=model_cfg['input_dim'],
    hidden_dims=model_cfg['hidden_dims'],
    output_dim=model_cfg['output_dim'],
    dropout=model_cfg.get('dropout', 0.1),
    activation=model_cfg.get('activation', 'gelu'),
    normalize_output=model_cfg.get('normalize_output', True),
).to(device)
mlp.load_state_dict(checkpoint['model_state_dict'])
mlp.eval()

with torch.no_grad():
    features_clip = mlp(features_3d)
    features_clip = F.normalize(features_clip, dim=-1)

out_path.parent.mkdir(parents=True, exist_ok=True)
np.save(out_path, features_clip.detach().cpu().numpy().astype(np.float32))
print('Saved CLIP-space features to:', out_path)
print('CLIP-space features shape:', tuple(features_clip.shape))
PY

In [ ]:
features_clip = np.load(FEATURES_CLIP_PATH).astype(np.float32)
print('Loaded cached CLIP-space features:', features_clip.shape)

## 3. Text Query to Heatmap

In [ ]:
import open_clip
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(DEVICE)
clip_model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')
features_clip_torch = torch.from_numpy(features_clip).to(DEVICE)
print('CLIP query model ready on', DEVICE)

In [ ]:
def query_scene(text_query: str, top_percent: float = 10.0):
    print(f'\nQuery: "{text_query}"')

    tokens = tokenizer([text_query]).to(DEVICE)
    with torch.no_grad():
        text_emb = clip_model.encode_text(tokens)
        text_emb = torch.nn.functional.normalize(text_emb, dim=-1)

    similarity = (features_clip_torch @ text_emb.T).squeeze(-1).cpu().numpy()
    sim_min, sim_max = similarity.min(), similarity.max()
    sim_norm = (similarity - sim_min) / (sim_max - sim_min + 1e-8)

    threshold = np.percentile(sim_norm, 100 - top_percent)
    highlight_mask = sim_norm >= threshold
    print(f'Highlighted points: {highlight_mask.sum():,} / {len(coord):,}')

    heatmap_colors = np.full((len(coord), 3), 0.6)
    heatmap_colors[highlight_mask] = [1.0, 0.2, 0.2]

    fig = go.Figure(data=[go.Scatter3d(
        x=coord[:, 0],
        y=coord[:, 1],
        z=coord[:, 2],
        mode='markers',
        marker=dict(
            size=1.5,
            color=[f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in heatmap_colors],
            opacity=0.8,
        )
    )])

    fig.update_layout(
        title=f'Query: "{text_query}" - top {top_percent}% matches highlighted in red',
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', bgcolor='rgb(20,20,20)'),
        paper_bgcolor='rgb(30,30,30)',
        font_color='white',
        height=700,
    )
    fig.show()
    return sim_norm

In [ ]:
sim = query_scene('chair', top_percent=10)

In [ ]:
sim = query_scene('table', top_percent=10)

In [ ]:
your_query = input('Enter your query (e.g. "chair", "window", "door"): ')
sim = query_scene(your_query, top_percent=10)